# Training data preparation and preflight

This tutorial prepares molecular spectra for learning. You will read named training/validation files, validate their chemistry and identity separation, choose model settings and inspect saved training/checkpoint results.

In [1]:
import json, os, sys, io, tempfile
from pathlib import Path
from IPython.display import display, SVG
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'documentation/assets').is_dir())
from openmolspec.dataset import MolecularSpectra, list_datasets
print('Registered datasets:', [c['name'] for c in list_datasets()])

Registered datasets: ['transpeaknet_hsqc_test', 'opendatalab_nmr', 'nmrspec_test', 'patent_test_10k', 'nmrexp_exp_10k', 'nmrgym_test', 'openspec_test', 'massspecgym_test', 'massspecgym_1_5', 'opendatalab_nmr_prepared', 'opendatalab_ir', 'opendatalab_19f', 'opendatalab_31p', 'opendatalab_ms', 'opendatalab_11b', 'opendatalab_raman', 'opendatalab_29si']


## 1. Load training and validation records

The example files contain two OpenDataLab/WanJuan molecules per split. Keep the split names when importing your own files so the preparation step can check molecular overlap. Each record supplies a structure, formula, spectral text and available-channel mask.

In [2]:
data = MolecularSpectra.from_jsonl(splits={s: root / f'documentation/assets/sdk-nmr-{s}-examples.jsonl' for s in ('train', 'val')}, name='documented_opendatalab_train_val', domain='experimental')
for role, view in data.get_split().items():
    print(role, [row['smiles'] for row in view])

train ['C#CCCc1ccccc1', 'CC=C=Cc1ccccc1']
val ['c1c2c(cc3c1CC3)CC2', 'C=C1Cc2ccccc2C1']


## 2. Prepare data and inspect a training plan

Preparation validates the structure/formula relationship and H/C nucleus labels. The small Transformer configuration below uses 32 hidden channels and a batch size of two. `validate` checks the prepared files; `describe` prints the command and selected settings.

In [3]:
from openmolspec.training import prepare_nmr_training
from openmolspec.model import NMRTransformer
prepared_tmp = tempfile.TemporaryDirectory(prefix='openmolspec-training-tutorial-')
prepared = prepare_nmr_training(data['train'], data['val'], Path(prepared_tmp.name) / 'data')
model = NMRTransformer(stage='align')
plan = model.training_plan(prepared, run_dir=Path(prepared_tmp.name) / 'run', epochs=1, batch_size=2, parameters={'hidden_size': 32, 'heads': 4, 'encoder_layers': 1, 'decoder_layers': 1, 'feedforward_size': 64, 'projection_size': 16, 'max_train_batches': 1, 'max_val_examples': 2})
print(json.dumps(plan.validate(), indent=2))
print(json.dumps(plan.describe(), indent=2))
assert not plan.run_dir.exists()

{
  "validated": true,
  "training_performed": false,
  "uses_test_split": false,
  "split_audit": {
    "identity": "connectivity",
    "nmr_chemistry_checked": true,
    "records": {
      "train": 2,
      "val": 2
    },
    "within_split_duplicates": {
      "train": 0,
      "val": 0
    },
    "overlaps": [
      {
        "left": "train",
        "right": "val",
        "molecules": 0
      }
    ],
    "disjoint": true,
    "scope": "supplied views only; filtered/limited views do not establish full-dataset independence"
  }
}
{
  "command": [
    "/mnt/shared-storage-user/xiejiaqing/OpenMolSpec/studio/.runtime/docs-sphinx/bin/python",
    "-m",
    "spec2struct_fork.train",
    "align",
    "--data-dir",
    "/tmp/openmolspec-training-tutorial-irhn2vjb/data",
    "--run-dir",
    "/tmp/openmolspec-training-tutorial-irhn2vjb/run",
    "--epochs",
    "1",
    "--batch-size",
    "2",
    "--learning-rate",
    "0.0001",
    "--seed",
    "2026",
    "--device",
    "cpu",
    "

## 3. Launch training and inspect saved checkpoints

Install the training dependencies and call `trained = model.fit(plan)` to launch the plan. The following cell reads the saved encoder and generator training report and checks checkpoint identity and save/reload prediction agreement.

In [4]:
roundtrip = json.loads((root / 'documentation/assets/reports/sdk_training_roundtrip_20261117.json').read_text())
assert roundtrip['completed'] and roundtrip['training_performed']
assert roundtrip['test_records'] == 0
for stage, result in roundtrip['stages'].items():
    assert result['save_reload_inference_equal']
    print(stage, 'training/save/reload passed', result['checkpoint_sha256'])
print('Verified training stages:', len(roundtrip['stages']))

align training/save/reload passed 13950f0306ec2a958f52daf2d8646e7a34794f62d615365911eca1c09fbf299a
generate training/save/reload passed 7458c141273390b6c5c21c2e61f4f32a1d90aad18a7df827ed7f3206868dcee0
Verified training stages: 2


[Python SDK guide](../api.md)


## References

- OpenDataLab. WanJuan MolSpectra, version 1.0.3. Experimental molecular spectroscopy dataset. [Source](https://opendatalab.com/)
- OpenMolSpec contributors. OpenMolSpec: Chemical spectroscopy datasets, models and learning workflows. Software release v1.0 (2026). [Source](https://github.com/jiaqingxie/openmolspec-studio-frontend/releases/tag/v1.0)
